# Tree-Based Models and Rolling-Origin Cross-Validation
### Teaching Notebook 03 | Data Science Education Series
**Author:** Md Minhazur Rahman 
**MSc Data Science (Merit)** — University of Greenwich 
**Preprint:** [DOI: 10.5281/zenodo.19479285](https://doi.org/10.5281/zenodo.19479285) 
**Live Research Demo:** [Streamlit Dashboard](https://retail-forecasting-hvdzvesi4u9l6fs5tvdoyi.streamlit.app/) 

---

> **Prerequisites:** Complete [Notebook 01](./01_intro_time_series_forecasting.ipynb) and [Notebook 02](./02_feature_engineering_for_time_series.ipynb) before starting this notebook.

> **Who is this for?**
> Students who understand linear models and feature engineering and are ready to learn why tree-based models dominate real-world forecasting — and how to evaluate them honestly using Rolling-Origin Cross-Validation.

---

## What You Will Learn

| Section | Topic | Key Concept |
|---|---|---|
| 1 | Why Linear Regression has limits | Decision boundaries and non-linearity |
| 2 | Random Forest | Ensemble learning through many trees |
| 3 | XGBoost | Sequential error correction |
| 4 | Rolling-Origin Cross-Validation | The gold standard for time-series evaluation |
| 5 | Full model comparison | Which model wins and why |
| 6 | From notebook to dashboard | How this becomes a deployed system |

---

## Section 1: Why Linear Regression Has Limits

Across Notebooks 01 and 02, we used Linear Regression as our model. Even with good features, it has a fundamental constraint:

> **Linear Regression assumes that the relationship between your features and your target is a straight line.**

In retail forecasting, this assumption breaks down constantly:

| Situation | Linear Assumption | Reality |
|---|---|---|
| Eid sales spike | Gradual increase | Sudden jump then sharp drop |
| Promotion effect | Proportional boost | Threshold effect — small discounts do nothing, 30% off causes a rush |
| Weather impact | Linear relationship | Non-linear — mild rain has no effect, heavy rain closes the shop |
| Stock-out effect | Continuous | Sales drop to zero abruptly |

These are **non-linear relationships**. Linear Regression cannot model them.

### The Solution: Tree-Based Models

Decision trees split data into regions based on rules:

```
IF lag_1 > 60 AND rolling_mean_7 > 55:
    predict HIGH sales
ELIF lag_7 < 40:
    predict LOW sales
ELSE:
    predict MEDIUM sales
```

This naturally captures non-linear patterns, threshold effects, and interactions between features — without any assumption about the shape of the relationship.

---

In [ ]:
# Import all libraries needed for this notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

# Reproduce the full feature-engineered dataset from Notebook 02
np.random.seed(42)
n_days = 100
day_index = np.arange(1, n_days + 1)

trend    = 0.3 * day_index
seasonal = 10 * np.sin(2 * np.pi * day_index / 7)
noise    = np.random.normal(0, 4, size=n_days)
sales    = 40 + trend + seasonal + noise

df = pd.DataFrame({'day': day_index, 'sales': sales})

# Feature engineering (from Notebook 02 — shift to prevent leakage)
df['sales_lag_1']     = df['sales'].shift(1)
df['sales_lag_7']     = df['sales'].shift(7)
df['rolling_mean_7']  = df['sales'].shift(1).rolling(7).mean()
df['rolling_std_7']   = df['sales'].shift(1).rolling(7).std()

df = df.dropna().copy()

FEATURES = ['day', 'sales_lag_1', 'sales_lag_7',
            'rolling_mean_7', 'rolling_std_7']
TARGET   = 'sales'

print('Dataset ready with feature engineering applied.')
print(f'Shape: {df.shape}')
print(f'Features: {FEATURES}')

## Section 2: Random Forest

### The Core Idea: Wisdom of the Crowd

A single decision tree is powerful but unstable — small changes in the training data can produce a completely different tree.

**Random Forest** solves this by building hundreds of decision trees, each trained on a slightly different random sample of the data and a random subset of features. The final prediction is the **average** of all trees.

```
Tree 1 predicts:  58.2 units
Tree 2 predicts:  61.4 units
Tree 3 predicts:  57.9 units
...               ...
Tree 100 predicts: 60.1 units

Final prediction: average = 59.4 units
```

### Why does averaging help?

Each individual tree makes errors — but the errors are **random and uncorrelated**. When you average many uncorrelated errors, they cancel each other out. What remains is the true signal.

This principle is called **variance reduction through ensembling** and it is one of the most powerful ideas in machine learning.

### Key Parameters

| Parameter | What It Controls | Typical Value |
|---|---|---|
| `n_estimators` | Number of trees | 100–500 |
| `max_depth` | How deep each tree grows | 3–10 |
| `min_samples_leaf` | Minimum samples per leaf | 1–10 |

In [ ]:
# ── Train Random Forest on a simple train-test split first ─────────────
split_point = 80
train = df[df['day'] <= split_point]
test  = df[df['day'] >  split_point]

# Train Random Forest
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=5,
    min_samples_leaf=3,
    random_state=42
)
rf_model.fit(train[FEATURES], train[TARGET])
rf_predictions = rf_model.predict(test[FEATURES])

# Evaluate
rf_mae  = mean_absolute_error(test[TARGET], rf_predictions)
rf_rmse = np.sqrt(mean_squared_error(test[TARGET], rf_predictions))
rf_mape = np.mean(np.abs(
    (test[TARGET] - rf_predictions) / test[TARGET]
)) * 100

print('── Random Forest Results ──')
print(f'MAE:   {rf_mae:.2f} units')
print(f'RMSE:  {rf_rmse:.2f} units')
print(f'MAPE:  {rf_mape:.2f}%')
print()

# Feature importance
importance = pd.Series(
    rf_model.feature_importances_,
    index=FEATURES
).sort_values(ascending=False)

print('── Feature Importance ──')
print(importance.to_string())
print()
print('Feature importance tells us which features the model')
print('relied on most heavily when making predictions.')

In [ ]:
# Visualise feature importance
plt.figure(figsize=(9, 4))
importance.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Random Forest — Feature Importance',
           fontsize=13, fontweight='bold')
plt.xlabel('Feature')
plt.ylabel('Importance Score')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print('Which feature does the model consider most important?')
print(f'Answer: {importance.index[0]}')
print()
print('This tells us something important about the structure of the data.')
print('The model has automatically discovered which features carry the most')
print('predictive signal — without us telling it.')

## Section 3: XGBoost

### The Core Idea: Learning From Mistakes

While Random Forest builds trees **in parallel** (independently of each other), XGBoost builds trees **sequentially** — each new tree specifically targets the errors made by all previous trees.

```
Round 1: Build Tree 1
         Predict: 58.0  |  Actual: 65.0  |  Error: +7.0

Round 2: Build Tree 2 to predict the ERROR of Tree 1
         Predict error: 6.5
         Combined prediction: 58.0 + 6.5 = 64.5

Round 3: Build Tree 3 to predict the REMAINING ERROR
         And so on...

Final: Sum of all trees = very close to 65.0
```

This process is called **gradient boosting** — each tree gradient-descends toward lower error.

### Random Forest vs XGBoost — When to Use Which?

| Situation | Prefer |
|---|---|
| Noisy data with many outliers | Random Forest |
| Clean data, need maximum accuracy | XGBoost |
| Fast training needed | Random Forest |
| Kaggle competition | XGBoost |
| Teaching beginners | Random Forest (simpler to explain) |
| Production retail forecasting | XGBoost (with careful tuning) |

In [ ]:
# ── Train XGBoost ──────────────────────────────────────────────────────
xgb_model = XGBRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0
)
xgb_model.fit(train[FEATURES], train[TARGET])
xgb_predictions = xgb_model.predict(test[FEATURES])

# Evaluate
xgb_mae  = mean_absolute_error(test[TARGET], xgb_predictions)
xgb_rmse = np.sqrt(mean_squared_error(test[TARGET], xgb_predictions))
xgb_mape = np.mean(np.abs(
    (test[TARGET] - xgb_predictions) / test[TARGET]
)) * 100

print('── XGBoost Results ──')
print(f'MAE:   {xgb_mae:.2f} units')
print(f'RMSE:  {xgb_rmse:.2f} units')
print(f'MAPE:  {xgb_mape:.2f}%')

## Section 4: Rolling-Origin Cross-Validation

### The Problem With a Single Train-Test Split

In Notebooks 01 and 02, we used one train-test split:
- Train: days 1–80
- Test: days 81–100

This gives us **one** estimate of model performance. But what if days 81–100 happened to be unusually easy or unusually hard to predict? Our single estimate could be misleading.

### The Solution: Rolling-Origin Cross-Validation (ROCV)

ROCV simulates the real forecasting process by making multiple predictions across different time windows:

```
Fold 1:  Train: days 1–60   |  Test: days 61–70
Fold 2:  Train: days 1–70   |  Test: days 71–80
Fold 3:  Train: days 1–80   |  Test: days 81–90
Fold 4:  Train: days 1–90   |  Test: days 91–100

Final MAE = average MAE across all 4 folds
```

**Critical rules:**
1. The training set always starts from the beginning — no gaps
2. The training set always ends BEFORE the test window begins — no leakage
3. Each fold moves the origin forward — hence rolling-origin

> This is the exact validation method used in my MSc dissertation. It is the reason the reported MAPE of 11.7% is a reliable, honest estimate of real-world performance — not an optimistic single-split figure.

### Why Does This Matter?

A model evaluated on one test split might look excellent simply because that period was easy. ROCV forces the model to prove itself across multiple different time periods — giving a much more honest picture of how it will perform in deployment.

In [ ]:
# ── Implement Rolling-Origin Cross-Validation from scratch ─────────────

def rolling_origin_cv(df, features, target, model,
                      min_train_size=60, step=10):
    """
    Rolling-Origin Cross-Validation for time-series.

    Parameters:
        df           : DataFrame with features and target
        features     : list of feature column names
        target       : target column name
        model        : sklearn-compatible model with fit/predict
        min_train_size: minimum number of rows in training set
        step         : number of rows in each test window

    Returns:
        results : list of dicts with fold metrics
    """
    results = []
    n = len(df)

    for fold_end in range(min_train_size + step, n + 1, step):
        train_end  = fold_end - step
        test_start = train_end
        test_end   = fold_end

        train_fold = df.iloc[:train_end]
        test_fold  = df.iloc[test_start:test_end]

        if len(test_fold) == 0:
            continue

        model.fit(train_fold[features], train_fold[target])
        predictions = model.predict(test_fold[features])

        mae  = mean_absolute_error(test_fold[target], predictions)
        rmse = np.sqrt(mean_squared_error(
            test_fold[target], predictions))
        mape = np.mean(np.abs(
            (test_fold[target] - predictions) /
            test_fold[target]
        )) * 100

        results.append({
            'fold':        len(results) + 1,
            'train_rows':  train_end,
            'test_start':  df.iloc[test_start]['day'],
            'test_end':    df.iloc[test_end - 1]['day'],
            'mae':         mae,
            'rmse':        rmse,
            'mape':        mape
        })

    return results


print('ROCV function defined.')
print('Running ROCV for all three models...')
print()

In [ ]:
# ── Run ROCV for all three models ──────────────────────────────────────

# Linear Regression
lr_results = rolling_origin_cv(
    df, FEATURES, TARGET,
    LinearRegression(),
    min_train_size=60, step=10
)

# Random Forest
rf_results = rolling_origin_cv(
    df, FEATURES, TARGET,
    RandomForestRegressor(
        n_estimators=200, max_depth=5,
        min_samples_leaf=3, random_state=42),
    min_train_size=60, step=10
)

# XGBoost
xgb_results = rolling_origin_cv(
    df, FEATURES, TARGET,
    XGBRegressor(
        n_estimators=200, max_depth=4,
        learning_rate=0.05, subsample=0.8,
        colsample_bytree=0.8, random_state=42,
        verbosity=0),
    min_train_size=60, step=10
)


def summarise_rocv(results, model_name):
    maes  = [r['mae']  for r in results]
    mapes = [r['mape'] for r in results]
    print(f'── {model_name} ──')
    print(f'  Folds evaluated:  {len(results)}')
    print(f'  Mean MAE:         {np.mean(maes):.2f} units')
    print(f'  Std  MAE:         {np.std(maes):.2f} units')
    print(f'  Mean MAPE:        {np.mean(mapes):.2f}%')
    print()


print('=== ROCV RESULTS ===')
print()
summarise_rocv(lr_results,  'Linear Regression')
summarise_rocv(rf_results,  'Random Forest')
summarise_rocv(xgb_results, 'XGBoost')

In [ ]:
# ── Visualise ROCV MAE across folds ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

folds = [r['fold'] for r in lr_results]

# Plot 1: MAE per fold
axes[0].plot(folds, [r['mae'] for r in lr_results],
             marker='o', label='Linear Regression',
             color='red', linewidth=1.8)
axes[0].plot(folds, [r['mae'] for r in rf_results],
             marker='s', label='Random Forest',
             color='steelblue', linewidth=1.8)
axes[0].plot(folds, [r['mae'] for r in xgb_results],
             marker='^', label='XGBoost',
             color='green', linewidth=1.8)
axes[0].set_title('MAE Per Fold\n(lower is better)',
                   fontweight='bold')
axes[0].set_xlabel('ROCV Fold')
axes[0].set_ylabel('MAE (units)')
axes[0].legend()

# Plot 2: Mean MAE comparison bar chart
model_names = ['Linear\nRegression', 'Random\nForest', 'XGBoost']
mean_maes   = [
    np.mean([r['mae'] for r in lr_results]),
    np.mean([r['mae'] for r in rf_results]),
    np.mean([r['mae'] for r in xgb_results])
]
colours = ['red', 'steelblue', 'green']

bars = axes[1].bar(model_names, mean_maes,
                    color=colours, edgecolor='white', width=0.5)
axes[1].set_title('Mean MAE Across All ROCV Folds\n(lower is better)',
                   fontweight='bold')
axes[1].set_ylabel('Mean MAE (units)')

for bar, val in zip(bars, mean_maes):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.2,
                 f'{val:.2f}', ha='center',
                 fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

## Section 5: Full Model Comparison

We have now evaluated three models using Rolling-Origin Cross-Validation — the most honest evaluation method for time-series forecasting.

The results tell a clear story:

### What The Numbers Mean

**Mean MAE** tells us: on average, how many units was each model wrong by per day?

**Std MAE** tells us: how consistent was the model across different time periods? A high standard deviation means the model is unreliable — it performs well in some periods and poorly in others.

For a retail business, **consistency matters as much as accuracy**. A model that is always 5 units wrong is more useful than one that is sometimes 2 units wrong and sometimes 15 units wrong.

### The Progression We Have Built

| Notebook | What We Added | What Improved |
|---|---|---|
| 01 | Baseline Linear Regression (day only) | Starting point |
| 02 | Feature engineering (lags + rolling) | Significant MAE reduction |
| 03 | Tree-based models + ROCV | Further improvement + honest evaluation |

This is the real machine learning workflow:
> **Start simple. Add features. Upgrade the model. Evaluate honestly. Repeat.**

---

## Section 6: From Notebook to Dashboard

Everything we have built across three notebooks — synthetic data, feature engineering, tree-based models, ROCV evaluation — is the foundation of a real deployed system.

### How a Jupyter Notebook Becomes a Production Tool

```
RESEARCH NOTEBOOK          PRODUCTION SYSTEM
──────────────────         ─────────────────────────────
np.random.seed(42)    →    Parametric data generator
df['sales_lag_1']     →    Automated feature pipeline
rf_model.fit(...)     →    Trained model saved to disk
plt.plot(...)         →    Interactive Streamlit charts
print(f'MAE: ...')    →    Live metric dashboard
```

### The Live System Built From This Pipeline

My MSc dissertation took exactly this progression — from research notebook to deployed interactive system — and the result is publicly available for you to explore:

**Try the live dashboard:** [https://retail-forecasting-hvdzvesi4u9l6fs5tvdoyi.streamlit.app/](https://retail-forecasting-hvdzvesi4u9l6fs5tvdoyi.streamlit.app/)

In the dashboard you can:
- Select any SKU and adjust the forecast horizon
- Toggle promotions and holidays to see scenario effects
- View model performance metrics in real time
- Compare predictions across model types

**The underlying code:** [github.com/minhazda/synthetic-retail-forecasting](https://github.com/minhazda/synthetic-retail-forecasting)

**The published research:** [DOI: 10.5281/zenodo.19479285](https://doi.org/10.5281/zenodo.19479285)

### For Students: Your Next Steps

You now have the foundation to build your own forecasting system. Here is a suggested project path:

1. **Modify Notebook 01** — change the seasonal period from 7 to 30 (monthly pattern)
2. **Modify Notebook 02** — add a lag-14 feature and measure whether it improves accuracy
3. **Modify Notebook 03** — increase `n_estimators` in Random Forest and observe the effect on ROCV MAE
4. **Extension project** — add a holiday spike to the synthetic data generator and build a model that handles it
5. **Advanced project** — replace the parametric generator with CTGAN (a deep learning synthetic data generator) and compare data fidelity

---

## Discussion Questions for Students

1. Random Forest averages 200 trees. What would happen if you used only 10 trees? Run the experiment and report the MAE change.

2. XGBoost has a `learning_rate` parameter. What happens when you set it to 0.5 instead of 0.05? Is higher always better?

3. Our ROCV used a step size of 10 days. What would change if you used a step size of 5 days? Would you trust the results more or less?

4. In a real retail system in Bangladesh — for example, a shop in Agrabad, Chittagong — what additional features would you add to this pipeline that we have not considered here?

5. The live dashboard was built with Streamlit. What are the advantages and disadvantages of Streamlit compared to a full web application framework like Django or Flask?

---

## Series Complete

You have completed the three-notebook Data Science Teaching Series:

| Notebook | Topic |
|---|---|
| [01 — Introduction to Time-Series Forecasting](./01_intro_time_series_forecasting.ipynb) | Synthetic data, Linear Regression, train-test split |
| [02 — Feature Engineering for Time-Series](./02_feature_engineering_for_time_series.ipynb) | Lag features, rolling windows, feature leakage |
| 03 — Tree-Based Models and ROCV | Random Forest, XGBoost, Rolling-Origin Cross-Validation |

---
*Teaching materials for undergraduate data science education in Bangladesh.*
*© Md Minhazur Rahman, 2026. Open for educational use.*